# 23CSE301 ML Capstone — EV Charging Stations

## Notebook Structure
1. **Dataset Understanding**
2. **Preprocessing**
3. **REGRESSIONS — 10 Algorithms**
4. **CLASSIFICATION — 5 Algorithms**

Preprocessing is kept separate so the prepared data can be reused by all models.

### Targets
- **Regression target:** `Oppurtunity_Score`
- **Classification target:** `Profit_Potential` (`Low` / `Medium` / `High`)

## 0. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
)
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_PATH = "EV_Charging_Profit_Classification_Raw_with_Opportunity_Score.xlsx"

REG_TARGET = "Oppurtunity_Score"
CLF_TARGET = "Profit_Potential"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. Dataset Understanding

In [3]:
df = pd.read_excel(DATA_PATH)

print("Shape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Shape: (65134, 23)
Rows: 65134
Columns: 23


,Station Name,City,State,ZIP,Status Code,EV Level1 EVSE Num,EV Level2 EVSE Num,EV DC Fast Count,EV Network,EV Connector Types,Latitude,Longitude,Owner Type Code,Open Date,Facility Type,EV Pricing,Restricted Access,EV Workplace Charging,Access Code,Nearest_Station_Distance_km,Competition_Level,Profit_Potential,Opportunity_Score
0,LADWP - Truesdale Center,Sun Valley,CA,91352,E,NaN,57.0,2.0,SHELL_RECHARGE,CHADEMO J1772 J1772COMBO,34.248319,-118.387971,LG,36448.0,UTILITY,NaN,NaN,1.0,private,0.738,Low,High,79.46
1,Los Angeles Convention Center,Los Angeles,CA,90015,E,NaN,7.0,NaN,Non-Networked,J1772,34.040539,-118.271387,P,34941.0,PARKING_GARAGE,Free; parking fee,0.0,0.0,public,0.287,Medium,Medium,66.09
2,LADWP - John Ferraro Building,Los Angeles,CA,90012,E,NaN,338.0,12.0,Non-Networked,CHADEMO J1772 J1772COMBO,34.059133,-118.248589,LG,36448.0,UTILITY,NaN,NaN,1.0,private,0.000,High,Low,2.35
3,LADWP - Haynes Power Plant,Long Beach,CA,90803,E,NaN,19.0,1.0,Non-Networked,CHADEMO J1772 J1772COMBO,33.759802,-118.096665,LG,43221.0,UTILITY,NaN,NaN,1.0,private,1.032,Low,High,83.53
4,LADWP - Harbor Generating Station,Wilmington,CA,90744,E,NaN,10.0,NaN,Non-Networked,J1772,33.770508,-118.265628,LG,36448.0,UTILITY,NaN,NaN,1.0,private,0.278,Medium,Medium,65.67


### 1.1 Structure and data types

In [4]:
structure = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Null": df.isna().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})

display(structure)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

,Column,Data Type,Non-Null,Null,Unique Values
0,Station Name,object,65133,1,62509
1,City,object,65129,5,6494
2,State,object,65120,14,52
3,ZIP,object,65133,1,11104
4,Status Code,object,65134,0,1
5,EV Level1 EVSE Num,float64,667,64467,37
6,EV Level2 EVSE Num,float64,56677,8457,73
7,EV DC Fast Count,float64,9364,55770,43
8,EV Network,object,65132,2,45
9,EV Connector Types,object,65098,36,28


Numeric columns: 10
Categorical columns: 13


### 1.2 Basic checks

In [5]:
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
display(
    df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Count")
)

print("\nNumeric summary:")
display(df[numeric_cols].describe().T)

print("\nRegression target:", REG_TARGET)
display(pd.to_numeric(df[REG_TARGET], errors="coerce").describe())

print("\nClassification target:", CLF_TARGET)
display(df[CLF_TARGET].value_counts(dropna=False))

Duplicate rows: 13

Missing values:


,Missing Count
EV Level1 EVSE Num,64467
EV DC Fast Count,55770
Restricted Access,54603
EV Pricing,52174
Facility Type,47537
Owner Type Code,46768
EV Level2 EVSE Num,8457
Open Date,79
EV Connector Types,36
State,14



Numeric summary:


,count,mean,std,min,25%,50%,75%,max
EV Level1 EVSE Num,667.0,4.379310,8.532664,1.000000,1.000000,2.000000,4.000000,90.000000
EV Level2 EVSE Num,56677.0,2.420382,3.273703,1.000000,2.000000,2.000000,2.000000,338.000000
EV DC Fast Count,9364.0,4.206002,5.058703,1.000000,1.000000,2.000000,6.000000,84.000000
Latitude,65134.0,37.816967,5.040287,-41.354100,34.040619,38.503573,41.488183,64.852466
Longitude,65134.0,-96.670801,19.528462,-164.848855,-117.927948,-92.807549,-78.963729,122.373276
Open Date,65055.0,44093.412728,1084.487782,34941.000000,43819.000000,44265.000000,44848.000000,45536.000000
Restricted Access,10531.0,0.108442,0.310952,0.000000,0.000000,0.000000,0.000000,1.000000
EV Workplace Charging,65133.0,0.021341,0.144519,0.000000,0.000000,0.000000,0.000000,1.000000
Nearest_Station_Distance_km,65134.0,1.317836,35.114911,0.000000,0.006000,0.077000,0.537000,6629.882000
Opportunity_Score,65134.0,50.000339,28.860299,2.350000,26.980000,50.020000,74.990000,100.000000



Regression target: EV Level2 EVSE Num


count    56677.000000
mean         2.420382
std          3.273703
min          1.000000
25%          2.000000
50%          2.000000
75%          2.000000
max        338.000000
Name: EV Level2 EVSE Num, dtype: float64


Classification target: Profit_Potential


Profit_Potential
Low       21713
High      21711
Medium    21710
Name: count, dtype: int64

## 2. Preprocessing

### Order
1. Remove exact duplicates
2. Remove rows with missing target
3. Feature engineering
4. Select useful predictors and avoid leakage
5. Train/test split
6. Detect numeric outliers using **IQR**
7. Cap outliers using training-derived IQR limits
8. Fill numeric missing values using **KNN Imputer**
9. Fill categorical missing values
10. **One-Hot Encode** categorical variables
11. **Standardize** features
12. Apply **PCA** for dimensionality reduction

> Preprocessing is fitted on the training data only to avoid data leakage.

### 2.1 Basic cleaning and feature engineering

In [6]:
data = df.copy()

before = len(data)
data = data.drop_duplicates().copy()
print("Duplicates removed:", before - len(data))

if "Open Date" in data.columns:
    open_date = pd.to_datetime(
        data["Open Date"],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    )
    data["Open Year"] = open_date.dt.year
    data["Station Age"] = 2024 - data["Open Year"]

if "EV Connector Types" in data.columns:
    data["Num Connector Types"] = (
        data["EV Connector Types"]
        .fillna("")
        .astype(str)
        .str.strip()
        .apply(lambda x: 0 if not x else len(set(x.split())))
    )

if "EV Network" in data.columns:
    data["Is Networked"] = (
        data["EV Network"]
        .fillna("Non-Networked")
        .astype(str)
        .str.lower()
        .ne("non-networked")
        .astype(int)
    )

if "EV Pricing" in data.columns:
    data["Has Pricing"] = data["EV Pricing"].notna().astype(int)

if "EV Workplace Charging" in data.columns:
    data["Has Workplace Charging"] = (
        pd.to_numeric(
            data["EV Workplace Charging"],
            errors="coerce"
        ).fillna(0) > 0
    ).astype(int)

print("Shape after cleaning:", data.shape)

Duplicates removed: 13
Shape after cleaning: (65121, 29)


### 2.2 Feature selection

In [21]:
reg_features = [
    c for c in [
        "Latitude", "Longitude", "Open Year", "Station Age",
        "Num Connector Types", "Is Networked", "Has Pricing",
        "Has Workplace Charging", "Access Code", "State",
        "Facility Type", "EV Network"
    ]
    if c in data.columns and c not in {
        REG_TARGET, "EV Level1 EVSE Num", "EV DC Fast Count"
    }
]

clf_features = [
    c for c in [
        "Latitude",
        "Longitude",
        "Open Year",
        "Station Age",
        "Num Connector Types",
        "Is Networked",
        "Has Pricing",
        "Has Workplace Charging",
        "EV Level1 EVSE Num",
        "EV Level2 EVSE Num",
        "EV DC Fast Count",
        "State",
        "EV Network",
        "Facility Type",
        "Owner Type Code",
        "Access Code",
        "Status Code",
        "Restricted Access"
    ]
    if c in data.columns and c not in {
        CLF_TARGET,
        "Nearest_Station_Distance_km",
        "Competition_Level",
        "Opportunity_Score"
    }
]

print("Regression features:")
print(reg_features)

print("\nClassification features:")
print(clf_features)

print("\nClassification target-leakage fields excluded:")
print([
    "Nearest_Station_Distance_km",
    "Competition_Level",
    "Opportunity_Score"
])

Regression features:
['Latitude', 'Longitude', 'Open Year', 'Station Age', 'Num Connector Types', 'Is Networked', 'Has Pricing', 'Has Workplace Charging', 'Access Code', 'State', 'Facility Type', 'EV Network']

Classification features:
['Latitude', 'Longitude', 'Open Year', 'Station Age', 'Num Connector Types', 'Is Networked', 'Has Pricing', 'Has Workplace Charging', 'EV Level1 EVSE Num', 'EV Level2 EVSE Num', 'EV DC Fast Count', 'State', 'EV Network', 'Facility Type', 'Owner Type Code', 'Access Code', 'Status Code', 'Restricted Access']

Classification target-leakage fields excluded:
['Nearest_Station_Distance_km', 'Competition_Level', 'Opportunity_Score']


### 2.3 Train/test split

In [22]:
reg_df = data[reg_features + [REG_TARGET]].copy()
reg_df[REG_TARGET] = pd.to_numeric(
    reg_df[REG_TARGET],
    errors="coerce"
)
reg_df = reg_df.dropna(subset=[REG_TARGET])

X_reg = reg_df[reg_features]
y_reg = reg_df[REG_TARGET]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

clf_df = data[clf_features + [CLF_TARGET]].dropna(
    subset=[CLF_TARGET]
).copy()

X_clf = clf_df[clf_features]
y_clf = clf_df[CLF_TARGET].astype(str)

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf,
    y_clf,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

print("Regression:", X_reg_train.shape, X_reg_test.shape)
print("Classification:", X_clf_train.shape, X_clf_test.shape)

Regression: (45332, 12) (11333, 12)
Classification: (52096, 18) (13025, 18)


### 2.4 Outlier detection — IQR

**IQR = Q3 − Q1**

Potential outlier = value `< Q1 − 1.5×IQR` or `> Q3 + 1.5×IQR`.

In [24]:
def iqr_report(X, numeric_columns):
    rows = []

    for col in numeric_columns:
        s = pd.to_numeric(X[col], errors="coerce")
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        count = ((s < low) | (s > high)).sum()

        rows.append({
            "Feature": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower Bound": low,
            "Upper Bound": high,
            "Outliers": int(count)
        })

    return pd.DataFrame(rows)

reg_num = X_reg_train.select_dtypes(
    include=np.number
).columns.tolist()

clf_num = X_clf_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Regression outliers")
display(iqr_report(X_reg_train, reg_num))

print("Classification outliers")
display(iqr_report(X_clf_train, clf_num))

Regression outliers


,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outliers
0,Latitude,34.029373,41.507052,7.477679,22.812855,52.723570,344
1,Longitude,-118.021451,-78.769417,39.252035,-176.899504,-19.891364,6
2,Open Year,2019.000000,2022.000000,3.000000,2014.500000,2026.500000,3166
3,Station Age,2.000000,5.000000,3.000000,-2.500000,9.500000,3166
4,Num Connector Types,1.000000,1.000000,0.000000,1.000000,1.000000,3076
5,Is Networked,1.000000,1.000000,0.000000,1.000000,1.000000,6738
6,Has Pricing,0.000000,0.000000,0.000000,0.000000,0.000000,9854
7,Has Workplace Charging,0.000000,0.000000,0.000000,0.000000,0.000000,1053


Classification outliers


,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outliers
0,Latitude,34.030670,41.469460,7.438790,22.872484,52.627645,383
1,Longitude,-117.926814,-78.957632,38.969182,-176.380587,-20.503858,7
2,Open Year,2019.000000,2022.000000,3.000000,2014.500000,2026.500000,3314
3,Station Age,2.000000,5.000000,3.000000,-2.500000,9.500000,3314
4,Num Connector Types,1.000000,1.000000,0.000000,1.000000,1.000000,6588
5,Is Networked,1.000000,1.000000,0.000000,1.000000,1.000000,7386
6,Has Pricing,0.000000,0.000000,0.000000,0.000000,0.000000,10377
7,Has Workplace Charging,0.000000,0.000000,0.000000,0.000000,0.000000,1116
8,EV Level1 EVSE Num,1.000000,3.000000,2.000000,-2.000000,6.000000,78
9,EV Level2 EVSE Num,2.000000,2.000000,0.000000,2.000000,2.000000,16329


### 2.5 IQR outlier capping

In [25]:
def fit_iqr_caps(X, numeric_columns):
    caps = {}

    for col in numeric_columns:
        s = pd.to_numeric(X[col], errors="coerce")
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        caps[col] = (
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr
        )

    return caps


def apply_iqr_caps(X, caps):
    X = X.copy()

    for col, (low, high) in caps.items():
        X[col] = pd.to_numeric(
            X[col],
            errors="coerce"
        ).clip(low, high)

    return X


reg_caps = fit_iqr_caps(X_reg_train, reg_num)
clf_caps = fit_iqr_caps(X_clf_train, clf_num)

X_reg_train = apply_iqr_caps(X_reg_train, reg_caps)
X_reg_test = apply_iqr_caps(X_reg_test, reg_caps)

X_clf_train = apply_iqr_caps(X_clf_train, clf_caps)
X_clf_test = apply_iqr_caps(X_clf_test, clf_caps)

print("IQR capping completed.")

IQR capping completed.


### 2.6 KNN Imputation + One-Hot Encoding + Standardization

In [26]:
reg_num = X_reg_train.select_dtypes(include=np.number).columns.tolist()
reg_cat = X_reg_train.select_dtypes(exclude=np.number).columns.tolist()

clf_num = X_clf_train.select_dtypes(include=np.number).columns.tolist()
clf_cat = X_clf_train.select_dtypes(exclude=np.number).columns.tolist()

for col in reg_cat:
    X_reg_train[col] = X_reg_train[col].apply(
        lambda x: str(x) if pd.notna(x) else np.nan
    )
    X_reg_test[col] = X_reg_test[col].apply(
        lambda x: str(x) if pd.notna(x) else np.nan
    )

for col in clf_cat:
    X_clf_train[col] = X_clf_train[col].apply(
        lambda x: str(x) if pd.notna(x) else np.nan
    )
    X_clf_test[col] = X_clf_test[col].apply(
        lambda x: str(x) if pd.notna(x) else np.nan
    )

def make_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        (
            "num",
            Pipeline([
                ("knn_imputer", KNNImputer(n_neighbors=3)),
                ("scaler", StandardScaler())
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            cat_cols
        )
    ])

reg_preprocessor = make_preprocessor(reg_num, reg_cat)
clf_preprocessor = make_preprocessor(clf_num, clf_cat)

print("Regression numeric:", reg_num)
print("Regression categorical:", reg_cat)
print("Classification numeric:", clf_num)
print("Classification categorical:", clf_cat)

Regression numeric: ['Latitude', 'Longitude', 'Open Year', 'Station Age', 'Num Connector Types', 'Is Networked', 'Has Pricing', 'Has Workplace Charging']
Regression categorical: ['Access Code', 'State', 'Facility Type', 'EV Network']
Classification numeric: ['Latitude', 'Longitude', 'Open Year', 'Station Age', 'Num Connector Types', 'Is Networked', 'Has Pricing', 'Has Workplace Charging', 'EV Level1 EVSE Num', 'EV Level2 EVSE Num', 'EV DC Fast Count', 'Restricted Access']
Classification categorical: ['State', 'EV Network', 'Facility Type', 'Owner Type Code', 'Access Code', 'Status Code']


### 2.7 Transform data

In [27]:
X_reg_train_p = reg_preprocessor.fit_transform(X_reg_train)
X_reg_test_p = reg_preprocessor.transform(X_reg_test)

X_clf_train_p = clf_preprocessor.fit_transform(X_clf_train)
X_clf_test_p = clf_preprocessor.transform(X_clf_test)

print("Regression processed shape:", X_reg_train_p.shape)
print("Classification processed shape:", X_clf_train_p.shape)

Regression processed shape: (45332, 162)
Classification processed shape: (52096, 181)


### 2.8 PCA

In [28]:
pca_reg = PCA(
    n_components=0.95,
    svd_solver="full"
)

pca_clf = PCA(
    n_components=0.95,
    svd_solver="full"
)

X_reg_train_pca = pca_reg.fit_transform(
    X_reg_train_p
)
X_reg_test_pca = pca_reg.transform(
    X_reg_test_p
)

X_clf_train_pca = pca_clf.fit_transform(
    X_clf_train_p
)
X_clf_test_pca = pca_clf.transform(
    X_clf_test_p
)

print("Regression PCA shape:",
      X_reg_train_pca.shape)

print("Classification PCA shape:",
      X_clf_train_pca.shape)

print(
    "Regression variance retained:",
    round(
        pca_reg.explained_variance_ratio_.sum(),
        4
    )
)

print(
    "Classification variance retained:",
    round(
        pca_clf.explained_variance_ratio_.sum(),
        4
    )
)

Regression PCA shape: (45332, 37)
Classification PCA shape: (52096, 37)
Regression variance retained: 0.9514
Classification variance retained: 0.9508


## 3. REGRESSIONS — 10 Algorithms

Use:
- `X_reg_train_pca`
- `X_reg_test_pca`
- `y_reg_train`
- `y_reg_test`

### Mandatory metrics
- R² Score
- RMSE
- MAE
- 5-Fold Cross-Validated R²

### 1. Linear Regression

In [ ]:
# ============================================
# 1. LINEAR REGRESSION
# ============================================

linear_model = LinearRegression()

# Fit
linear_model.fit(X_reg_train_pca, y_reg_train)

# Predict
y_pred_linear = linear_model.predict(X_reg_test_pca)

# Metrics
r2_linear = r2_score(y_reg_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_reg_test, y_pred_linear))
mae_linear = mean_absolute_error(y_reg_test, y_pred_linear)

print("LINEAR REGRESSION")
print("-" * 40)
print(f"R²   : {r2_linear:.4f}")
print(f"RMSE : {rmse_linear:.4f}")
print(f"MAE  : {mae_linear:.4f}")

# Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_reg_test, y_pred_linear, alpha=0.5)
plt.plot(
    [y_reg_test.min(), y_reg_test.max()],
    [y_reg_test.min(), y_reg_test.max()],
    linestyle="--"
)
plt.xlabel("Actual Opportunity Score")
plt.ylabel("Predicted Opportunity Score")
plt.title("Linear Regression - Actual vs Predicted")
plt.tight_layout()
plt.show()

# Residual Plot
residuals_linear = y_reg_test - y_pred_linear

plt.figure(figsize=(7, 5))
plt.scatter(y_pred_linear, residuals_linear, alpha=0.5)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Opportunity Score")
plt.ylabel("Residuals")
plt.title("Linear Regression - Residual Plot")
plt.tight_layout()
plt.show()

### 2. Ridge Regression

In [ ]:
# ============================================
# 2. RIDGE REGRESSION
# ============================================

ridge_model = Ridge(alpha=1.0)

# Fit
ridge_model.fit(X_reg_train_pca, y_reg_train)

# Predict
y_pred_ridge = ridge_model.predict(X_reg_test_pca)

# Metrics
r2_ridge = r2_score(y_reg_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_reg_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_reg_test, y_pred_ridge)

print("RIDGE REGRESSION")
print("-" * 40)
print(f"R²   : {r2_ridge:.4f}")
print(f"RMSE : {rmse_ridge:.4f}")
print(f"MAE  : {mae_ridge:.4f}")

# Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_reg_test, y_pred_ridge, alpha=0.5)
plt.plot(
    [y_reg_test.min(), y_reg_test.max()],
    [y_reg_test.min(), y_reg_test.max()],
    linestyle="--"
)
plt.xlabel("Actual Opportunity Score")
plt.ylabel("Predicted Opportunity Score")
plt.title("Ridge Regression - Actual vs Predicted")
plt.tight_layout()
plt.show()

# Residual Plot
residuals_ridge = y_reg_test - y_pred_ridge

plt.figure(figsize=(7, 5))
plt.scatter(y_pred_ridge, residuals_ridge, alpha=0.5)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Opportunity Score")
plt.ylabel("Residuals")
plt.title("Ridge Regression - Residual Plot")
plt.tight_layout()
plt.show()

### 3. Lasso Regression

In [ ]:
# ============================================
# 3. LASSO REGRESSION
# ============================================

lasso_model = Lasso(
    alpha=0.01,
    max_iter=10000
)

# Fit
lasso_model.fit(X_reg_train_pca, y_reg_train)

# Predict
y_pred_lasso = lasso_model.predict(X_reg_test_pca)

# Metrics
r2_lasso = r2_score(y_reg_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_reg_test, y_pred_lasso))
mae_lasso = mean_absolute_error(y_reg_test, y_pred_lasso)

print("LASSO REGRESSION")
print("-" * 40)
print(f"R²   : {r2_lasso:.4f}")
print(f"RMSE : {rmse_lasso:.4f}")
print(f"MAE  : {mae_lasso:.4f}")

# Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_reg_test, y_pred_lasso, alpha=0.5)
plt.plot(
    [y_reg_test.min(), y_reg_test.max()],
    [y_reg_test.min(), y_reg_test.max()],
    linestyle="--"
)
plt.xlabel("Actual Opportunity Score")
plt.ylabel("Predicted Opportunity Score")
plt.title("Lasso Regression - Actual vs Predicted")
plt.tight_layout()
plt.show()

# Residual Plot
residuals_lasso = y_reg_test - y_pred_lasso

plt.figure(figsize=(7, 5))
plt.scatter(y_pred_lasso, residuals_lasso, alpha=0.5)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Opportunity Score")
plt.ylabel("Residuals")
plt.title("Lasso Regression - Residual Plot")
plt.tight_layout()
plt.show()

### 4. ElasticNet Regression

In [ ]:
# ============================================
# 4. ELASTIC NET REGRESSION
# ============================================

elastic_model = ElasticNet(
    alpha=0.01,
    l1_ratio=0.5,
    max_iter=10000,
    random_state=42
)

# Fit
elastic_model.fit(X_reg_train_pca, y_reg_train)

# Predict
y_pred_elastic = elastic_model.predict(X_reg_test_pca)

# Metrics
r2_elastic = r2_score(y_reg_test, y_pred_elastic)
rmse_elastic = np.sqrt(mean_squared_error(y_reg_test, y_pred_elastic))
mae_elastic = mean_absolute_error(y_reg_test, y_pred_elastic)

print("ELASTIC NET REGRESSION")
print("-" * 40)
print(f"R²   : {r2_elastic:.4f}")
print(f"RMSE : {rmse_elastic:.4f}")
print(f"MAE  : {mae_elastic:.4f}")

# Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_reg_test, y_pred_elastic, alpha=0.5)
plt.plot(
    [y_reg_test.min(), y_reg_test.max()],
    [y_reg_test.min(), y_reg_test.max()],
    linestyle="--"
)
plt.xlabel("Actual Opportunity Score")
plt.ylabel("Predicted Opportunity Score")
plt.title("ElasticNet Regression - Actual vs Predicted")
plt.tight_layout()
plt.show()

# Residual Plot
residuals_elastic = y_reg_test - y_pred_elastic

plt.figure(figsize=(7, 5))
plt.scatter(y_pred_elastic, residuals_elastic, alpha=0.5)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Opportunity Score")
plt.ylabel("Residuals")
plt.title("ElasticNet Regression - Residual Plot")
plt.tight_layout()
plt.show()

### 5. Polynomial Regression

In [ ]:
# ============================================
# 5. POLYNOMIAL REGRESSION
# ============================================

# Create polynomial features
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_reg_train_poly = poly.fit_transform(X_reg_train_pca)
X_reg_test_poly = poly.transform(X_reg_test_pca)

# Linear Regression on polynomial features
poly_model = LinearRegression()

# Fit
poly_model.fit(X_reg_train_poly, y_reg_train)

# Predict
y_pred_poly = poly_model.predict(X_reg_test_poly)

# Metrics
r2_poly = r2_score(y_reg_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_reg_test, y_pred_poly))
mae_poly = mean_absolute_error(y_reg_test, y_pred_poly)

print("POLYNOMIAL REGRESSION")
print("-" * 40)
print("Degree : 2")
print(f"R²     : {r2_poly:.4f}")
print(f"RMSE   : {rmse_poly:.4f}")
print(f"MAE    : {mae_poly:.4f}")

# Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_reg_test, y_pred_poly, alpha=0.5)
plt.plot(
    [y_reg_test.min(), y_reg_test.max()],
    [y_reg_test.min(), y_reg_test.max()],
    linestyle="--"
)
plt.xlabel("Actual Opportunity Score")
plt.ylabel("Predicted Opportunity Score")
plt.title("Polynomial Regression - Actual vs Predicted")
plt.tight_layout()
plt.show()

# Residual Plot
residuals_poly = y_reg_test - y_pred_poly

plt.figure(figsize=(7, 5))
plt.scatter(y_pred_poly, residuals_poly, alpha=0.5)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Opportunity Score")
plt.ylabel("Residuals")
plt.title("Polynomial Regression - Residual Plot")
plt.tight_layout()
plt.show()

### Comparison Table for first 5 Regression algorithms

In [ ]:
# ============================================
# REGRESSION MODEL COMPARISON
# ============================================

comparison_df = pd.DataFrame({
    "Algorithm": [
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "ElasticNet Regression",
        "Polynomial Regression"
    ],
    "R²": [
        r2_linear,
        r2_ridge,
        r2_lasso,
        r2_elastic,
        r2_poly
    ],
    "RMSE": [
        rmse_linear,
        rmse_ridge,
        rmse_lasso,
        rmse_elastic,
        rmse_poly
    ],
    "MAE": [
        mae_linear,
        mae_ridge,
        mae_lasso,
        mae_elastic,
        mae_poly
    ]
})

# Rank models by R²
comparison_df = comparison_df.sort_values(
    by="R²",
    ascending=False
).reset_index(drop=True)

# Round values for display
comparison_df["R²"] = comparison_df["R²"].round(4)
comparison_df["RMSE"] = comparison_df["RMSE"].round(4)
comparison_df["MAE"] = comparison_df["MAE"].round(4)

print("Regression Model Comparison - Opportunity Score")
display(comparison_df)

### 6. Decision Tree Regressor

### 7. Random Forest Regressor

### 8. Gradient Boosting Regressor

### 9. Support Vector Regressor (SVR)

### 10. K-Nearest Neighbors Regressor

## 4. CLASSIFICATION — 5 Algorithms

### Classification target
**`Profit_Potential` = Low / Medium / High**

Use:
- `X_clf_train_pca`
- `X_clf_test_pca`
- `y_clf_train`
- `y_clf_test`

### Mandatory metrics
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC-AUC

### 1. Logistic Regression

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

logistic_model.fit(
    X_clf_train_pca,
    y_clf_train
)

y_pred_logistic = logistic_model.predict(
    X_clf_test_pca
)

y_prob_logistic = logistic_model.predict_proba(
    X_clf_test_pca
)

print("Accuracy:",
      round(accuracy_score(y_clf_test, y_pred_logistic), 4))

print("Precision:",
      round(
          precision_score(
              y_clf_test,
              y_pred_logistic,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("Recall:",
      round(
          recall_score(
              y_clf_test,
              y_pred_logistic,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("F1 Score:",
      round(
          f1_score(
              y_clf_test,
              y_pred_logistic,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("ROC-AUC:",
      round(
          roc_auc_score(
              y_clf_test,
              y_prob_logistic,
              multi_class="ovr"
          ), 4
      ))

print("\nClassification Report:")
print(
    classification_report(
        y_clf_test,
        y_pred_logistic,
        zero_division=0
    )
)

print("Confusion Matrix:")
display(
    pd.DataFrame(
        confusion_matrix(
            y_clf_test,
            y_pred_logistic
        ),
        index=logistic_model.classes_,
        columns=logistic_model.classes_
    )
)

### 2. K-Nearest Neighbors

In [ ]:
knn_model = KNeighborsClassifier(
    n_neighbors=7,
    weights="distance"
)

knn_model.fit(
    X_clf_train_pca,
    y_clf_train
)

y_pred_knn = knn_model.predict(
    X_clf_test_pca
)

y_prob_knn = knn_model.predict_proba(
    X_clf_test_pca
)

print("Accuracy:",
      round(accuracy_score(y_clf_test, y_pred_knn), 4))

print("Precision:",
      round(
          precision_score(
              y_clf_test,
              y_pred_knn,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("Recall:",
      round(
          recall_score(
              y_clf_test,
              y_pred_knn,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("F1 Score:",
      round(
          f1_score(
              y_clf_test,
              y_pred_knn,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("ROC-AUC:",
      round(
          roc_auc_score(
              y_clf_test,
              y_prob_knn,
              multi_class="ovr"
          ), 4
      ))

print("\nClassification Report:")
print(
    classification_report(
        y_clf_test,
        y_pred_knn,
        zero_division=0
    )
)

print("Confusion Matrix:")
display(
    pd.DataFrame(
        confusion_matrix(
            y_clf_test,
            y_pred_knn
        ),
        index=knn_model.classes_,
        columns=knn_model.classes_
    )
)

### 3. Gaussian Naive Bayes

In [ ]:
nb_model = GaussianNB()

nb_model.fit(
    X_clf_train_pca,
    y_clf_train
)

y_pred_nb = nb_model.predict(
    X_clf_test_pca
)

y_prob_nb = nb_model.predict_proba(
    X_clf_test_pca
)

print("Accuracy:",
      round(accuracy_score(y_clf_test, y_pred_nb), 4))

print("Precision:",
      round(
          precision_score(
              y_clf_test,
              y_pred_nb,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("Recall:",
      round(
          recall_score(
              y_clf_test,
              y_pred_nb,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("F1 Score:",
      round(
          f1_score(
              y_clf_test,
              y_pred_nb,
              average="weighted",
              zero_division=0
          ), 4
      ))

print("ROC-AUC:",
      round(
          roc_auc_score(
              y_clf_test,
              y_prob_nb,
              multi_class="ovr"
          ), 4
      ))

print("\nClassification Report:")
print(
    classification_report(
        y_clf_test,
        y_pred_nb,
        zero_division=0
    )
)

print("Confusion Matrix:")
display(
    pd.DataFrame(
        confusion_matrix(
            y_clf_test,
            y_pred_nb
        ),
        index=nb_model.classes_,
        columns=nb_model.classes_
    )
)

### 4. Decision Tree Classifier

### 5. Support Vector Machine (SVC)

## 5. Classification Model Comparison

## 6. ML Confidence Score

## Preprocessing Checklist

- Dataset structure and data types
- Missing-value inspection
- Duplicate removal
- Target definition
- Target leakage control
- Train/test split
- IQR outlier detection
- IQR outlier capping
- KNN imputation for numeric values
- Categorical imputation
- One-hot encoding
- Standardization
- PCA
- 10 separate regression algorithm cells
- 5 separate classification algorithm cells
- Classification model comparison
- ML confidence score